# ForkEvent pipeline test (`08_fork_events`)

This notebook tests `08_fork_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_fork`. It runs the real notebook with `%run` and `test_schema = test_fork`, so **production tables are never written**. Section B15 proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `08_fork_events` (they create `silver.fork_events` and `quarantine.quarantine_fork_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

ForkEvent keeps no event-specific fields, so this test covers the shared pipeline: JSON parse error, missing or invalid core IDs, warnings, blank-text normalization, duplicates, rerun idempotency, all-dates mode, a partial write failure and recovery. It also checks that the content of `payload` (including `forkee`) never affects a ForkEvent.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_fork"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("fork_events")
TEST_QUARANTINE = T("quarantine_fork_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.fork_events",
    "github_lakehouse.quarantine.quarantine_fork_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 08_fork_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.fork_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_fork_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Dry run against the empty test tables

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="2024-02-01" $run_id="test-dry-run"

## B4. Dry-run checks: proves production was not touched

In [0]:
check("resolved bronze", BRONZE_TABLE, TEST_BRONZE)
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("resolved audit", AUDIT_TABLE, TEST_AUDIT)

# An empty MERGE still commits a new version,
# so any write that leaked to production would show up here.
for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])

check_audit({"bronze_input_rows": 0, "silver_rows_inserted": 0,
             "quarantine_rows_inserted": 0}, expected_status="NO_INPUT")
check("audit pipeline_name", audit["pipeline_name"], "silver_fork_events")
check("audit event_type", audit["event_type"], "ForkEvent")
check("test audit rows", spark.table(TEST_AUDIT).count(), 1)
finish("B4 dry run")

## B5. Load the synthetic Bronze data

| Row | Case | Expected result |
|---|---|---|
| `f1` | clean ForkEvent | Silver, no warning |
| `""` | blank `event_id` | quarantine `MISSING: event_id` |
| `f3` (twice) | `repo_id = "abc"`; the same line ingested twice | one quarantine row `INVALID_CAST: repo_id`, newest ingestion kept |
| `f4` | `actor.id` missing | Silver, warning `MISSING: actor_id` |
| `f7` | `actor` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `f8` | `payload` is text, not an object | Silver, no warning (payload is out of scope) |
| `f9` | no `payload` at all | Silver, no warning |
| `f10` | an IssuesEvent | ignored (not read by this notebook) |
| `f11` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `f15` | bad `created_at` | Silver, warning `INVALID_CAST: event_timestamp` |
| `f17` | `repo.id` missing | quarantine `MISSING: repo_id` |
| `f19` | `actor.login` is blank | Silver, `actor_login` NULL, warning `MISSING: actor_login` |
| `f13` | NULL `_source_date` | only read in all-dates mode |
| `f14` | `_source_date` 2024-02-02 | only read in all-dates mode |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1, D2 = date(2024, 2, 1), date(2024, 2, 2)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

def ev(event_id, type_="ForkEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, payload=None, drop=()):
    """One GitHub-style event as a JSON string. Defaults = a clean ForkEvent."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": ({"forkee": {"id": 999, "full_name": "alice/repo"}}
                    if payload is None else payload),
    }
    for key in drop:
        event.pop(key)
    return json.dumps(event)

f3_raw = ev("f3", repo={"id": "abc", "name": "octo/repo"})

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)              # case
    (ev("f1"),                                            F1, D1, I1),   # clean
    (ev(""),                                              F1, D1, I1),   # blank event_id -> quarantine
    (f3_raw,                                              F1, D1, I1),   # repo_id "abc" -> quarantine
    (f3_raw,                                              F1, D1, I2),   # same line re-ingested -> 1 quarantine row
    (ev("f4", actor={"login": "bob"}),                    F1, D1, I1),   # actor_id missing -> warn
    (ev("f7", actor="oops"),                              F1, D1, I1),   # actor not an object -> parse error
    (ev("f8", payload="oops"),                            F1, D1, I1),   # payload not an object -> still clean
    (ev("f9", drop=["payload"]),                          F1, D1, I1),   # no payload -> still clean
    (ev("f10", type_="IssuesEvent"),                      F1, D1, I1),   # other event type -> ignored
    (ev("f11", actor={"id": 111, "login": "old"}),        F1, D1, I1),   # duplicate, older
    (ev("f11", actor={"id": 112, "login": "new"}),        F2, D1, I2),   # duplicate, newer -> wins
    (ev("f15", created_at="not-a-date"),                  F1, D1, I1),   # bad timestamp -> warn
    (ev("f17", repo={"name": "octo/repo"}),               F1, D1, I1),   # repo_id missing -> quarantine
    (ev("f19", actor={"id": 119, "login": "   "}),        F1, D1, I1),   # blank login -> NULL + warn
    (ev("f13"), "synthetic/unknown.json.gz", None, I1),                 # NULL _source_date
    (ev("f14", created_at="2024-02-02T10:00:00Z"),
        "synthetic/2024-02-02-0.json.gz", D2, I1),                       # other date
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 16)
finish("B5 load")

## B6. Run 1: single date 2024-02-01

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="2024-02-01" $run_id="test-run1"

## B7. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 1, "null_actor_id": 1, "null_actor_login": 1,
    "null_repo_name": 0, "missing_lineage": 0,
}

# --- audit: 13 ForkEvent rows on 2024-02-01 (f10 is an IssuesEvent; f13/f14 are other dates)
check_audit({
    "bronze_input_rows": 13, "hard_null_quarantine_rows": 5,
    "validation_quarantine_rows": 0, "validated_rows": 8,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 7,
    "silver_rows_inserted": 7, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 4, "quarantine_duplicates_removed": 1,
    "quarantine_rows_inserted": 4,
}, QUALITY_RUN1)
check("processing_scope", audit["processing_scope"], "SINGLE_DATE")

# --- Silver schema: exactly the contract columns, no payload columns
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      ["f1", "f11", "f15", "f19", "f4", "f8", "f9"])
check("silver only ForkEvent",
      spark.table(TEST_SILVER).where("event_type <> 'ForkEvent'").count(), 0)

r = one_silver("f1")
check("f1 ids", (r.get("actor_id"), r.get("repo_id")), (101, 201))
check("f1 labels", (r.get("actor_login"), r.get("repo_name")), ("alice", "octo/repo"))
check("f1 event_date", r.get("event_date"), D1)
f1_ts = (spark.table(TEST_SILVER).where("event_id = 'f1'")
         .selectExpr("date_format(event_timestamp, 'yyyy-MM-dd HH:mm:ss') AS ts")
         .collect())
check("f1 timestamp (UTC)", f1_ts[0]["ts"] if f1_ts else None, "2024-02-01 10:00:00")
check("f1 warning", r.get("quarantine_warning"), None)

r = one_silver("f4")
check("f4 actor_id", r.get("actor_id"), None)
check("f4 warning", r.get("quarantine_warning"), "MISSING: actor_id")

for event_id in ["f8", "f9"]:
    r = one_silver(event_id)
    check(f"{event_id} warning (payload ignored)", r.get("quarantine_warning"), None)

r = one_silver("f11")
check("f11 winner = newer ingestion", (r.get("actor_id"), r.get("_source_file")), (112, F2))

r = one_silver("f15")
check("f15 timestamp/date", (r.get("event_timestamp"), r.get("event_date")), (None, None))
check("f15 warning", r.get("quarantine_warning"), "INVALID_CAST: event_timestamp")

r = one_silver("f19")
check("f19 actor_login", r.get("actor_login"), None)
check("f19 warning", r.get("quarantine_warning"), "MISSING: actor_login")

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine rows", len(q), 4)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q),
      sorted(["MISSING: event_id", "INVALID_CAST: repo_id",
              "JSON_PARSE_ERROR", "MISSING: repo_id"]))
check("quarantine keys unique & non-NULL",
      len({r["quarantine_key"] for r in q if r["quarantine_key"]}), 4)
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantined_at stamped", all(r["quarantined_at"] for r in q), True)

f3 = (spark.table(TEST_QUARANTINE).where("event_id = 'f3'")
      .selectExpr("date_format(_ingested_at, 'yyyy-MM-dd') AS ingested", "repo_id_raw")
      .collect())
check("f3 collapsed to one row", len(f3), 1)
check("f3 keeps newest ingestion", f3[0]["ingested"] if f3 else None, "2024-03-02")
check("f3 raw value kept", f3[0]["repo_id_raw"] if f3 else None, "abc")

finish("B7 run 1")

## B8. Run 2: rerun the same date

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="2024-02-01" $run_id="test-run2"

## B9. Run 2 checks: idempotency

In [0]:
check_audit({
    "bronze_input_rows": 13, "silver_candidate_rows": 7,
    "silver_rows_inserted": 0, "silver_rows_already_present": 7,
    "quarantine_candidate_rows": 4, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 7)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 4)
finish("B9 run 2")

## B10. Run 3: all dates `blank source_date`

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="" $run_id="test-run3"

## B11. Run 3 checks: backfill scope and the NULL-date row

In [0]:
check_audit({
    "bronze_input_rows": 15, "hard_null_quarantine_rows": 5,
    "validated_rows": 10, "duplicate_rows_removed": 1,
    "silver_candidate_rows": 9, "silver_rows_inserted": 2,
    "silver_rows_already_present": 7, "quarantine_candidate_rows": 4,
    "quarantine_rows_inserted": 0,
}, {**QUALITY_RUN1, "missing_lineage": 1})
check("processing_scope", audit["processing_scope"], "ALL_DATES")
check("source_date", audit["source_date"], None)

r = one_silver("f13")
check("f13 _source_date", r.get("_source_date"), None)
check("f13 warning", r.get("quarantine_warning"), "MISSING: _source_date")
r = one_silver("f14")
check("f14 event_date", r.get("event_date"), D2)

check("silver rows", spark.table(TEST_SILVER).count(), 9)
check("silver count = distinct event_id",
      spark.table(TEST_SILVER).select("event_id").distinct().count(), 9)

# Unit test of the defensive event-type rule from the validation cell.
# Its input is pre-filtered to ForkEvent, so it cannot be reached with real data.
rule = (spark.createDataFrame([("ForkEvent",), ("IssuesEvent",), (None,)], "event_type string")
        .select("event_type", invalid_event_type.alias("r")).collect())
check("invalid_event_type rule",
      {x["event_type"]: x["r"] for x in rule},
      {"ForkEvent": None, "IssuesEvent": "INVALID_EVENT_TYPE", None: None})
finish("B11 run 3")

## B12. Partial-failure setup, then Run 4 `this cell is EXPECTED to fail`

In [0]:
D3 = date(2024, 2, 3)
F3 = "synthetic/2024-02-03-0.json.gz"
(spark.createDataFrame(
    [(ev("f16", created_at="2024-02-03T10:00:00Z"), F3, D3, I1),          # clean
     (ev("f18", created_at="2024-02-03T10:00:00Z",
         repo={"id": "xyz", "name": "octo/repo"}), F3, D3, I1)],         # bad repo_id
    spark.table(TEST_BRONZE).schema)
 .write.mode("append").saveAsTable(TEST_BRONZE))

# Make ONLY the Silver insert of f16 fail, after the quarantine MERGE has committed.
spark.sql(f"ALTER TABLE {TEST_SILVER} ADD CONSTRAINT block_f16 CHECK (event_id <> 'f16')")
print("Failure injected on", TEST_SILVER)

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="2024-02-03" $run_id="test-run4-fail"

## B13. Run 4 checks: partial failure state, then remove the fault

In [0]:
check("quarantine write", fork_write_results["quarantine"]["status"], "SUCCEEDED")
check("quarantine inserted", fork_write_results["quarantine"]["rows_inserted"], 1)
check("silver write", fork_write_results["silver"]["status"], "FAILED")
check("error mentions constraint",
      "block_f16" in (fork_write_results["silver"].get("error") or "").lower(), True)
check_audit({"quarantine_rows_inserted": 1, "silver_rows_inserted": None},
            expected_status="FAILED", reconciled=False)
check("silver unchanged after failure", spark.table(TEST_SILVER).count(), 9)
check("quarantine has f18", spark.table(TEST_QUARANTINE).count(), 5)

spark.sql(f"ALTER TABLE {TEST_SILVER} DROP CONSTRAINT block_f16")
finish("B13 run 4 (failure)")

## B14. Run 5: recovery rerun of the same date

In [0]:
%run ../08_fork_events $test_schema="test_fork" $source_date="2024-02-03" $run_id="test-run5-recover"

## B15. Run 5 checks, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 2, "hard_null_quarantine_rows": 1,
    "validated_rows": 1, "silver_candidate_rows": 1,
    "silver_rows_inserted": 1, "quarantine_candidate_rows": 1,
    "quarantine_rows_inserted": 0,          # f18 was already written by the failed run
})
check("silver rows", spark.table(TEST_SILVER).count(), 10)
check("quarantine rows", spark.table(TEST_QUARANTINE).count(), 5)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history",
      statuses, ["NO_INPUT", "SUCCEEDED", "SUCCEEDED", "SUCCEEDED", "FAILED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B15 run 5 (recovery) + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")